# Nile-Chat-4B — Query Router / Intent-Classification Sidecar Server

Serves `MBZUAI-Paris/Nile-Chat-4B` as an OpenAI-compatible `/v1/chat/completions`
endpoint via vLLM, dedicated **only** to `IntentRoutingController`'s
`classify_and_rewrite()` call (see `src/stores/query_router/providers/NileChat4BProvider.py`)
— never used for Mode A reply generation, which stays on the separately-deployed
fine-tuned 12B checkpoint.

**Runtime**: Runtime → Change runtime type → **A100 GPU** (paid tier). Nile-Chat-4B
is Gemma3-architecture (`model_type: gemma3_text`). An earlier version of this
notebook targeted free-tier T4 and hit a real, confirmed vLLM dead end there: vLLM
refuses `float16` for Gemma3-family models outright (`ValidationError: ... does not
support float16. Reason: Numerical instability`), and separately refuses
`bfloat16` on T4 because its compute capability (7.5) is below the 8.0 vLLM
requires — leaving no working dtype on that hardware. **A100 has compute
capability 8.0**, so `bfloat16` — the dtype Gemma3 was actually trained/published
in — works natively here. That's the whole reason this notebook now requires A100
specifically, not a smaller/cheaper GPU tier.

## 1. Confirm the GPU Colab actually gave you

In [ ]:
!nvidia-smi


**Check the output above before continuing.** This notebook requires an A100
(compute capability 8.0) — confirm the GPU name shown is actually `A100`, not `T4`
or `L4` (which is also below the 8.0 threshold `bfloat16` needs here). If you got a
different GPU, either request an A100 runtime again or see the dtype explanation
above for why the vLLM path in this notebook won't work on anything below compute
capability 8.0 for this specific model.

## 2. Install vLLM

In [1]:
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -qU uv
!uv pip install --system vllm --torch-backend=auto


Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 124.8 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 196 packages in 2.16s
Prepared 101 packages in 1m 10s
Uninstalled 14 packages in 153ms
Installed 101 packages in 274ms
 + agent-detector==1.1.0
 + anthropic==1.2.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cud

In [2]:
import torch

torch_version = torch.__version__.split("+")[0]
torch_cuda = torch.version.cuda
cuda_tag = "cu" + torch_cuda.replace(".", "")

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print(\'torch:\', torch.__version__, torch.version.cuda); "
     "print(\'torchvision:\', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable.")


Detected torch==2.13.0 built for CUDA 13.2 -> installing matched torchvision from index cu132, leaving torchaudio uninstalled
torch: 2.13.0+cu132 13.2
torchvision: 0.28.0+cu132
torch/torchvision aligned and importable.


In [3]:
!nohup vllm serve "MBZUAI-Paris/Nile-Chat-12B" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8002 \
    --served-model-name nile-chat-12b-base \
    > vllm.log 2>&1 &

Poll until the server is up, then a raw smoke test. If this cell reports `server ready: False`, check the `vllm.log` tail printed above it before continuing — it now breaks out as soon as an error appears in the log instead of waiting the full timeout.

In [4]:
import time

ready = False
for attempt in range(60):  # up to 10 minutes -- first run also downloads ~8GB of weights
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log or "ValidationError" in log:
        print("vLLM logged an error while loading -- check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 60 vllm.log
print("\n--- server ready:", ready, "---\n")

!curl -s http://localhost:8002/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"nile-chat-12b-base","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'


[10s] still loading...
[20s] still loading...
[30s] still loading...
[40s] still loading...
[50s] still loading...
[60s] still loading...
[70s] still loading...
[80s] still loading...
[90s] still loading...
[100s] still loading...
[110s] still loading...
[120s] still loading...
[130s] still loading...
[140s] still loading...
[150s] still loading...
[160s] still loading...
[170s] still loading...
[180s] still loading...
[190s] still loading...
[200s] still loading...
[210s] still loading...
[220s] still loading...
[230s] still loading...
[240s] still loading...
[250s] still loading...
[260s] still loading...
[270s] still loading...
[280s] still loading...
[290s] still loading...
[300s] still loading...
[310s] still loading...
[320s] still loading...
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:06<00:00,  1.31s/it]
(EngineCore pid=2968) 
(EngineCore pid=2968) INFO 09-01 16:16:25 [default_loader.py:430] Loading weights took 6.66 seconds
(EngineCore pid=2968) INFO 09-01 

If the curl call above didn't return a real completion, stop and fix it before
opening a tunnel. Common causes at this stage: confirm you actually got an A100
(section 1) — `bfloat16` will hard-fail on anything below compute capability 8.0 —
or an OOM (lower `--gpu-memory-utilization`, though unlikely for a 4B model on an
A100).

## 5. Expose via Cloudflare Tunnel

Using **cloudflared**, not ngrok/localtunnel — this is a deliberate match to this
project's own already-proven pattern (every other model deployed this project used
cloudflared identically), and it needs no signup/authtoken, unlike ngrok's free
tier. If you specifically want ngrok instead, swap this cell for the ngrok
equivalent — but there's no reliability reason to.

In [5]:
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed -- re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8002 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s -- check cloudflared.log for errors and re-run this cell"

print("=" * 70)
print("Tunnel is live. Paste these EXACT lines into your local src/.env:")
print("=" * 70)
print(f"QUERY_ROUTER_BASE_URL={tunnel_url}")
print("QUERY_ROUTER_MODEL_NAME=nile-chat-12b")
print("QUERY_ROUTER_BACKEND=NILE_CHAT_4B")
print("QUERY_ROUTER_REQUEST_TIMEOUT_SECONDS=15")
print("=" * 70)
print("Then restart your local app process so main.py\'s startup_span picks up")
print("the new QUERY_ROUTER_BASE_URL and builds app.query_router_client.")
print("=" * 70)


Tunnel is live. Paste these EXACT lines into your local src/.env:
QUERY_ROUTER_BASE_URL=https://cube-from-endorsement-publishers.trycloudflare.com
QUERY_ROUTER_MODEL_NAME=nile-chat-12b
QUERY_ROUTER_BACKEND=NILE_CHAT_4B
QUERY_ROUTER_REQUEST_TIMEOUT_SECONDS=15
Then restart your local app process so main.py's startup_span picks up
the new QUERY_ROUTER_BASE_URL and builds app.query_router_client.


## 6. Verify with the exact `classify_and_rewrite` prompt shape

Not a generic "hi" ping — this sends the same real scenario this project's own
multi-turn testing has used throughout (short reply "اه" after the assistant asked
a specific branch question), through the same message structure
`NileChat4BProvider.classify_and_rewrite` actually builds, so you can see with your
own eyes whether the model returns clean `{"intent": ..., "resolved_query": ...}`
JSON or collapses into something else **before** wiring it into the real app.

In [10]:
import json
import re
import requests

_JSON_OBJECT_RE = re.compile(r"\{[^{}]*\}")

allowed_intents = [
    "complaint", "book_appointment", "cancel_appointment",
    "query_schedule", "query_price", "query_branch", "general_inquiry", "unclassified",
]

system_prompt = "\n".join([
    "You are a senior intent classifier and query-rewriting assistant for an Egyptian Arabic "
    "medical-services WhatsApp assistant.",
    "First, in a <reasoning>...</reasoning> block, briefly reason in 1-2 sentences about what "
    "the patient is actually asking for semantically, and whether the final message on its own "
    "is already a complete, standalone question or depends on the conversation above it.",
    "After the </reasoning> block, on a new line, output a single JSON object of the exact "
    "shape: {\"intent\": \"<one value>\", \"resolved_query\": \"<string>\"} and nothing else "
    "after it. Never wrap it in a code fence, never output it more than once, never add any "
    "text after it.",
    f"<one value> MUST be exactly one of: {json.dumps(allowed_intents, ensure_ascii=False)}",
    "resolved_query: rewrite the final patient message into a short, self-contained, standalone "
    "version that makes sense with NO prior conversation attached. If the final patient message "
    "is already a complete, standalone question or statement, resolved_query is that same "
    "message unchanged. Example: assistant asked \'تحب تعرف مواعيد فرع المهندسين؟\' and the "
    "patient replied \'اه\' -> resolved_query is \'مواعيد فرع المهندسين\', not \'اه\'.",
])

user_content = "\n".join([
    "## Recent conversation (context only):",
    "Assistant: تحب تعرف مواعيد فرع المهندسين؟",
    "",
    "## Final patient message to classify and rewrite:",
    "اه",
])

response = requests.post(
    "http://localhost:8002/v1/chat/completions",
    json={
        "model": "nile-chat-4b",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        "temperature": 0.0,
        "max_tokens": 220,
        "stop": ["<end_of_turn>"],
    },
    timeout=30,
)
response.raise_for_status()
raw = response.json()["choices"][0]["message"]["content"].strip()

print("--- RAW MODEL OUTPUT ---")
print(raw)
print("------------------------\n")

matches = _JSON_OBJECT_RE.findall(raw)
if not matches:
    print("FAIL: no JSON object found in the response at all.")
else:
    try:
        parsed = json.loads(matches[-1])
        intent = parsed.get("intent")
        resolved_query = parsed.get("resolved_query")
        print(f"Parsed intent: {intent!r}")
        print(f"Parsed resolved_query: {resolved_query!r}")
        ok = (
            intent in allowed_intents
            and isinstance(resolved_query, str)
            and resolved_query.strip()
            and resolved_query.strip() != "اه"
        )
        print("\nPASS: model correctly resolved the short reply." if ok else
              "\nCHECK CAREFULLY: parsed, but doesn\'t look right (see values above).")
    except (json.JSONDecodeError, AttributeError) as e:
        print(f"FAIL: found a JSON-shaped object but it didn\'t parse cleanly ({e}).")


HTTPError: 404 Client Error: Not Found for url: http://localhost:8002/v1/chat/completions

## 7. Keep this running

- The tunnel URL is **ephemeral** — it changes every time this notebook's runtime
  restarts or disconnects. When that happens, re-run from the serving cell
  onward, get a **new** URL, and update `QUERY_ROUTER_BASE_URL` in `src/.env`
  again before restarting your local app.
- This is a development/testing setup, not a production deployment.
- Leave this tab open and the runtime connected for as long as you want the sidecar
  reachable from your local app. Remember an A100 Colab runtime is billed while
  it's alive — shut it down when you're done testing.